## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path; 
import os
import subprocess

exercise = "day4_afternoon_selection_scans"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

In [ ]:
work_dir=${KENYA2026_WORK_DIR:-$HOME/kenya2026}
cd "$work_dir/day4_afternoon_selection_scans"
printf "Bash working directory: %s\n" "$PWD"

In [ ]:
work_dir <- Sys.getenv("KENYA2026_WORK_DIR", unset = file.path(path.expand("~"), "kenya2026"))
setwd(file.path(work_dir, "day4_afternoon_selection_scans"))
cat("R working directory:", getwd(), "\n")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.

# Exercise I: Frequency-based methods
----
## Variability statistics

We will use human individuals in the 1000 genomes project. Because of computational demands we will only be looking at a 20Mb region for a small subset of the individuals. We will use 56 African individuals (YRI) and 60 European individuals (CEU). These are the individuals what overlap with the HapMap project where many selection scans in humans have been performed. We will try to explore the *LCT* gene located at position 136.6 Mb.

Aim: locate the *LCT* selection signal using genotype data

We will use `selscan` to calculate nucleotide diversity from a prepared VCF. Let's start by setting the paths to the data and program:

In [ ]:
## path for this exercise
ThePath=/course/popgen24/cindy/selectionExercises
  
#selscan program folder
SS=/course/popgen24/cindy/selectionExercises/prog/selscan-linux-1.3.0/

# CEU VCF for the regional diversity analysis
ceuVCF=human/ceuLCT.recode.vcf

Let's inspect one VCF record and identify how the diploid genotypes are represented:

In [ ]:
tail -n 1 $ThePath/ceuLCT.recode.vcf

Each diploid genotype contains two alleles: `0` denotes the reference allele and `1` the alternate allele. A vertical pipe (`|`) indicates that the genotype is phased, whereas a slash (`/`) indicates unknown phase. The diversity calculation below uses allele counts and does not require phase information.

### Tajima's Theta (Tajima's π or nucleotide diversity)
---
First lets look at the variability of the data in the CEU.

- If there has been positive selection at the LCT loci what do you expect?

With that in mind then try to estimate Tajima's theta (pi) in 10k windows using selscan using the following command:

In [ ]:


$SS/selscan --pi --vcf $ceuVCF --pmap --out ceuLCT --threads 8 --pi-win 10000

Let's try to have look at the last 10 lines of the output file (which are named after the window size):

In [ ]:
tail -n 10 ceuLCT.pi.10000bp.out

- What is being output here?

Now try to plot the results in R using e.g. the following command:

In [ ]:
r<-read.table("ceuLCT.pi.10000bp.out",head=F)
r<-subset(r,V3!=0)
plot(r[,1]/1e6,r[,3],ylab="pi",xlab="Position (Mb)",pch=20)
causalSNP <- 136608646/1e6
abline(v=causalSNP,col="red")
legend("topleft",lty=1,col="red","LCT")

- Does the LCT have visibly reduced variability? Why?

To determine whether it is extreme we can compare with the pi values in rest of the region using the following R code:

In [ ]:
#continue in R

hist(r[,3],br=100, main="Histogram of pi values for all windows", xlab="pi")
causalWin<-subset(r,V1<(causalSNP*1e6)& V2>(causalSNP*1e6))
abline(v=causalWin[,3],col="red")
legend("topright",lty=1,col="red","LCT")
print(paste("The pi value in the window with LCT is",causalWin[,3]))

- Is the variability in the *LCT* region extreme?

### How much does the window size matter?

The original analysis used **10 kb windows**. We will now compare it with two alternative choices:

- **1 kb windows**
- **1 Mb windows**

Before running the cells, predict how changing the window size might affect the result. Then run both cells below; the commands use different output names, so they will not overwrite the original 10 kb result.


In [ ]:
# Alternative window sizes for comparison with the original 10 kb analysis.
# Notice the TWO things changed in each command: --pi-win and --out.

# 1 kb windows
$SS/selscan --pi --vcf $ceuVCF --pmap \
  --out ceuLCT_1kb --threads 8 --pi-win 1000

# 1 Mb windows
$SS/selscan --pi --vcf $ceuVCF --pmap \
  --out ceuLCT_1Mb --threads 8 --pi-win 1000000

The output filename contains the window size. The next cell reads the new **1 kb** and **1 Mb** files as well as the original **10 kb** file. Because this version of `selscan` reports diversity summed across each window, the code divides by the window width before comparing them. This gives π per base pair and puts all three plots on the same scale.

In [ ]:
# The filenames must match the --out and --pi-win values used above.
pi_1kb_raw  <- read.table("ceuLCT_1kb.pi.1000bp.out", header=FALSE)
pi_10kb_raw <- read.table("ceuLCT.pi.10000bp.out", header=FALSE)
pi_1Mb_raw  <- read.table("ceuLCT_1Mb.pi.1000000bp.out", header=FALSE)

# Keep only the 20 Mb interval covered by this VCF. selscan also writes empty
# windows between position 1 and the beginning of the data interval.
keep_LCT_interval <- function(x) subset(x, V2 > 120e6 & V1 <= 140e6)
pi_1kb_raw  <- keep_LCT_interval(pi_1kb_raw)
pi_10kb_raw <- keep_LCT_interval(pi_10kb_raw)
pi_1Mb_raw  <- keep_LCT_interval(pi_1Mb_raw)

# First count windows for which selscan returned zero inside the data interval.
window_summary <- data.frame(
  window_size = c("1 kb", "10 kb", "1 Mb"),
  total_windows = c(nrow(pi_1kb_raw), nrow(pi_10kb_raw), nrow(pi_1Mb_raw)),
  zero_windows = c(sum(pi_1kb_raw$V3 == 0),
                   sum(pi_10kb_raw$V3 == 0),
                   sum(pi_1Mb_raw$V3 == 0))
)
window_summary$percent_zero <- round(100 * window_summary$zero_windows /
                                     window_summary$total_windows, 1)
window_summary

# As above, omit zero-valued windows from the plots.
pi_1kb  <- subset(pi_1kb_raw, V3 != 0)
pi_10kb <- subset(pi_10kb_raw, V3 != 0)
pi_1Mb  <- subset(pi_1Mb_raw, V3 != 0)

# selscan reports the diversity summed within a window. Divide by window
# width so that the three window sizes can be compared fairly.
pi_1kb$pi_per_bp  <- pi_1kb$V3 / 1000
pi_10kb$pi_per_bp <- pi_10kb$V3 / 10000
pi_1Mb$pi_per_bp  <- pi_1Mb$V3 / 1000000

causalSNP_mb <- 136608646 / 1e6
y_limits <- range(c(pi_1kb$pi_per_bp, pi_10kb$pi_per_bp,
                    pi_1Mb$pi_per_bp), finite=TRUE)
old_par <- par(no.readonly=TRUE)
par(mfrow=c(3, 1), mar=c(4, 4, 2, 1))

plot(pi_1kb$V1 / 1e6, pi_1kb$pi_per_bp, ylim=y_limits, pch=20, cex=0.35,
     xlab="Position (Mb)", ylab="pi per bp",
     main="1 kb windows")
abline(v=causalSNP_mb, col="red")

plot(pi_10kb$V1 / 1e6, pi_10kb$pi_per_bp, ylim=y_limits, pch=20, cex=0.55,
     xlab="Position (Mb)", ylab="pi per bp",
     main="10 kb windows: original analysis")
abline(v=causalSNP_mb, col="red")

plot(pi_1Mb$V1 / 1e6, pi_1Mb$pi_per_bp, ylim=y_limits, pch=20,
     xlab="Position (Mb)", ylab="pi per bp",
     main="1 Mb windows")
abline(v=causalSNP_mb, col="red")

par(old_par)

Discuss the comparison:

- Which window size gives the noisiest result or the most zero-valued windows?
- Which window size loses the most spatial detail around *LCT*?
- Which of the three sizes gives the most useful balance between noise and resolution?
- Why might a very broad window dilute a local signal of selection?

## Population genetic differentiation statistics
----

### $F_{ST}$ and Population Branch Statistics (PBS)

Now that we've tried out single-population statistics, lets see how a selection scan that compares two populations performs.

The data for Hudson's $F_{ST}$ comparing CEU (Europeans) and YRI (West Africans) for the *LCT* region has already been pre-ran using 10000bp window sizes.


Let's copy the data into your folder and plot the results in `R`:

In [ ]:
r <- read.table("human/comboLCT.fst.out",header=T)
causalSNP <- 136608646
#plot FST results
plot(r$POS,r$FST,ylab="Fst");
abline(v=causalSNP,col="red")

Similarly, data for PBS comparing CEU and CHB (Han Chinese) to YRI for the *LCT* region has been performed with window sizes consisting of 75 variants per window (approximately 10000 bp). 

Let's plot the pre-ran PBS results in `R`:


In [ ]:
r <- read.table("human/comboLCT.pbs.out",header=T)
causalSNP <- 136608646
#plot PBS results
plot(r$POS,r$PBS,ylab="PBS");
abline(v=causalSNP,col="red")

**- How do both of these selection methods (Fst and PBS) compare to Tajima's π?**
 <br/><br/>
 <br/><br/>

### Quick check: frequency-based selection statistics

Run the following cell for a short quiz before moving from the regional examples to a whole-genome PBS scan.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/selection_scans_quizzes/human_frequency_methods.json')

# Exercise II: Whole-genome PBS with 1000 Genomes

Let's explore the human genome using PBS. First copy the precomputed allele-frequency information from the 1000 Genomes Project.

In [ ]:
cp -r -sf  /course/popgen24/anders/selectionScan/ ~/
#enter the folder you just copied
cd ~/selectionScan



Load the data and plotting functions into R. You do not need to understand all of this setup code.

In [ ]:

## load data and function
setwd("~/selectionScan/")

source("server.R")
winSize <- 50000
shinyDir<- "~/selectionScan/tmp/"

shinyPBS<-paste0(shinyDir,"pbs") 
shinyCSV<-paste0(shinyDir,"pbs.csv")


First select three populations:

    NAT - Native Americans (PERU+Mexico)
    CHB – East Asian - Han Chinese
    CEU – Central Europeans
    YRI – African - Nigerians

The first population defines the branch being investigated; the other two provide the comparisons. Choose CEU first, followed by CHB and YRI.

In [ ]:
#"NAT", "CEU", "CHB","YRI"

#### choose populations
#pops 1=NAT,2=CHB",3=CEU",4=YRI
myPops <- c(3,2,4)

First, get an overview of PBS across the whole genome with a Manhattan plot.


In [ ]:
options(repr.plot.width=17, repr.plot.height=8)
#### choose populations
PBSmanPlot(myPops)


Note which chromosomes have extreme values. A high PBS value means a long estimated branch length for a given population. We can then inspect one chromosome in more detail.

Choose the chromosome with the highest PBS value and set the starting position to `-1` to display the whole chromosome.

e.g.

# see entire chromosome 1


In [ ]:
options(repr.plot.width=17, repr.plot.height=10)
#modify below code with your selected chromsome
PBSmanRegion(myPops,chrom=1,start=-1)

Zoom in to the peak by changing start and end position in the code above.

# see region between 20Mb and 21Mb on chromosome 1


In [ ]:
options(repr.plot.width=17, repr.plot.height=10)

#modify below code
PBSmanRegion(myPops,chrom=5,start=33, end=34)


- Locate the most extreme regions of the genome and zoom in on them using the code above.
 - Identify the gene with the highest PBS value.
 - What does the gene do?
 - Try the LCT gene (the mutations are locate in the adjacent MCM6 gene). We can use a genome browser to find its genome position, e.g. https://genome-euro.ucsc.edu/cgi-bin/hgGateway (choose human GRCh37/hg19 genome).

If you have time you can try other genes. Note that there are some populations that you cannot test because the populations are not represented in the data, e.g. Tibetan, Ethiopian , Inuit, Siberians.

---

# Bonus exercise: transferring Fst and PBS to wildebeest

The human exercises used known candidate loci and mostly pre-generated selection statistics. Here we will calculate the same frequency-based statistics ourselves in a non-model species, using **9 homogeneous black wildebeest**, **10 homogeneous northern blue wildebeest collected in the Maasai Mara**, and **5 homogeneous southern blue/B-Etosha wildebeest**. The Maasai Mara animals are a locality-level subset of the article's W-Serengeti population; exact sample IDs are in `selection_wildebeest_bonus/popfile.tsv`.

To keep the exercise quick, we will analyse unthinned `HiC_scaffold_1` in **non-overlapping 100 kb windows**. It contains 1,056,342 variants (7.1% of the whole-genome dataset), including the broad differentiation peak discussed in the [wildebeest population-genomics article](https://www.nature.com/articles/s41467-024-47015-y). Our starting point is the prepared, indexed 24-sample VCF rather than the original 8.2 GB file. Pre-generated whole-genome scans are supplied after the runnable exercise.

In [ ]:
set -euo pipefail
EXERCISE_DIR=${KENYA2026_WORK_DIR:-$HOME/kenya2026}/day4_afternoon_selection_scans; DATA_DIR=${EXERCISE_DIR}/selection_wildebeest_bonus
RUN_DIR=${EXERCISE_DIR}/wildebeest_bonus_run
SOURCE_VCF=${DATA_DIR}/black_mara_etosha.vcf.gz
POPFILE=${DATA_DIR}/popfile.tsv
CHR1_VCF=${RUN_DIR}/black_mara_etosha.scaffold1.vcf.gz
COUNTS=${RUN_DIR}/scaffold1.population_allele_counts.tsv
mkdir -p ${RUN_DIR}

# Make the scaffold 1 working VCF from the prepared 24-sample dataset.
bcftools view --threads 4 -r HiC_scaffold_1 \
  -Oz -o ${CHR1_VCF} ${SOURCE_VCF}
bcftools index --threads 4 -f -t ${CHR1_VCF}

# Count reference and alternate alleles separately for all three populations.
bcftools +fill-tags ${CHR1_VCF} -Ou -- \
  -S ${POPFILE} -t AC,AN | \
bcftools query \
  -f '%POS\t%INFO/AN_black\t%INFO/AC_black\t%INFO/AN_maasai_mara\t%INFO/AC_maasai_mara\t%INFO/AN_b_etosha\t%INFO/AC_b_etosha\n' \
  > ${COUNTS}

bcftools index -n ${CHR1_VCF}
wc -l ${COUNTS}

## 1. Calculate pairwise Hudson Fst

For each variant, let $d_{between}$ be the mean pairwise difference between two populations and $d_{within}$ the mean of their within-population pairwise differences. The per-site Hudson components are

$$N=d_{between}-d_{within}, \qquad D=d_{between}.$$

Within each 100 kb window we estimate $F_{ST}=\sum N/\sum D$. Summing numerator and denominator before taking the ratio is important: averaging individual site-level Fst values is not equivalent. Variants exactly on a 100 kb upper boundary are omitted to reproduce the linked upstream implementation.

In [ ]:
from pathlib import Path; import numpy as np
import pandas as pd

run_dir = Path(__import__('os').environ.get('KENYA2026_WORK_DIR', Path.home() / 'kenya2026')) / 'day4_afternoon_selection_scans' / 'wildebeest_bonus_run'
counts_path = run_dir / 'scaffold1.population_allele_counts.tsv'
window_size = 100_000

count_columns = [
    'pos',
    'an_black', 'ac_black',
    'an_maasai_mara', 'ac_maasai_mara',
    'an_b_etosha', 'ac_b_etosha',
]
ac_columns = [name for name in count_columns if name.startswith('ac_')]
sites = pd.read_csv(
    counts_path, sep='\t', header=None, names=count_columns,
    dtype={name: 'string' for name in ac_columns},
)

# This filtered VCF is biallelic. AC is therefore one alternate-allele count.
if sites[ac_columns].apply(lambda x: x.str.contains(',', regex=False).any()).any():
    raise ValueError('This teaching implementation expects biallelic variants.')
sites[ac_columns] = sites[ac_columns].replace('.', '0').apply(pd.to_numeric)
max_position = int(sites['pos'].max())

# Match the interval convention used by the upstream wildebeest script.
sites = sites[sites['pos'] % window_size != 0].copy()
sites['window'] = (sites['pos'] - 1) // window_size
pairs = [
    ('black', 'maasai_mara'),
    ('black', 'b_etosha'),
    ('maasai_mara', 'b_etosha'),
]

def hudson_components(frame, population_1, population_2):
    n1 = frame[f'an_{population_1}']
    x1 = frame[f'ac_{population_1}']
    n2 = frame[f'an_{population_2}']
    x2 = frame[f'ac_{population_2}']
    valid = (n1 >= 2) & (n2 >= 2)

    with np.errstate(divide='ignore', invalid='ignore'):
        within_1 = 2 * x1 * (n1 - x1) / (n1 * (n1 - 1))
        within_2 = 2 * x2 * (n2 - x2) / (n2 * (n2 - 1))
        between = ((n1 - x1) * x2 + x1 * (n2 - x2)) / (n1 * n2)

    numerator = (between - (within_1 + within_2) / 2).where(valid)
    denominator = between.where(valid)
    return numerator, denominator, valid

for population_1, population_2 in pairs:
    label = f'{population_1}__{population_2}'
    numerator, denominator, valid = hudson_components(
        sites, population_1, population_2
    )
    sites[f'num_{label}'] = numerator
    sites[f'den_{label}'] = denominator
    sites[f'valid_{label}'] = valid

grouped = sites.groupby('window', sort=True)
window_index = pd.RangeIndex(max_position // window_size, name='window')
fst_windows = pd.DataFrame(index=window_index)
fst_windows['n_variants'] = grouped.size().reindex(window_index, fill_value=0)

for population_1, population_2 in pairs:
    label = f'{population_1}__{population_2}'
    numerator = grouped[f'num_{label}'].sum().reindex(window_index)
    denominator = grouped[f'den_{label}'].sum().reindex(window_index)
    all_valid = grouped[f'valid_{label}'].all().reindex(window_index, fill_value=False)
    fst_windows[f'fst_{label}'] = (numerator / denominator).where(
        all_valid & denominator.ne(0)
    )

fst_windows = fst_windows.reset_index()
fst_windows.insert(0, 'scaffold', 'HiC_scaffold_1')
fst_windows['start'] = fst_windows['window'] * window_size + 1
fst_windows['end'] = (fst_windows['window'] + 1) * window_size
fst_windows['midpoint'] = fst_windows['start'] + window_size / 2
fst_windows.to_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t', index=False)

fst_columns = [f'fst_{a}__{b}' for a, b in pairs]
pd.DataFrame({
    'median': fst_windows[fst_columns].median(),
    '99th percentile on scaffold 1': fst_windows[fst_columns].quantile(0.99),
    'maximum': fst_windows[fst_columns].max(),
})

## 2. Transform pairwise Fst into PBS branches

For three populations A, B, and C, transform each pairwise estimate as $T_{AB}=-\log(1-F_{ST,AB})$ and calculate

$$PBS_A=\frac{T_{AB}+T_{AC}-T_{BC}}{2}.$$

Finite negative Fst estimates are set to zero before transformation. PBS itself is **not** clipped: negative branch lengths are informative because they show that the three distances are not perfectly tree-like.

In [ ]:
from pathlib import Path; import numpy as np
import pandas as pd

run_dir = Path(__import__('os').environ.get('KENYA2026_WORK_DIR', Path.home() / 'kenya2026')) / 'day4_afternoon_selection_scans' / 'wildebeest_bonus_run'
fst_windows = pd.read_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t')
pbs_windows = fst_windows.copy()

def fst_distance(values):
    clamped = values.clip(lower=0, upper=1 - 1e-12)
    return -np.log1p(-clamped)

t_black_mara = fst_distance(pbs_windows['fst_black__maasai_mara'])
t_black_etosha = fst_distance(pbs_windows['fst_black__b_etosha'])
t_mara_etosha = fst_distance(pbs_windows['fst_maasai_mara__b_etosha'])

pbs_windows['pbs_black'] = (t_black_mara + t_black_etosha - t_mara_etosha) / 2
pbs_windows['pbs_maasai_mara'] = (t_black_mara + t_mara_etosha - t_black_etosha) / 2
pbs_windows['pbs_b_etosha'] = (t_black_etosha + t_mara_etosha - t_black_mara) / 2
pbs_windows.to_csv(run_dir / 'scaffold1.pbs.tsv', sep='\t', index=False)

branch_rows = []
for branch in ['black', 'maasai_mara', 'b_etosha']:
    column = f'pbs_{branch}'
    values = pbs_windows[column].dropna()
    best_index = values.idxmax()
    branch_rows.append({
        'branch': branch,
        'median': values.median(),
        '99th percentile on scaffold 1': values.quantile(0.99),
        'maximum': values.max(),
        'maximum window (Mb)': (
            f"{pbs_windows.loc[best_index, 'start'] / 1e6:.1f}-"
            f"{pbs_windows.loc[best_index, 'end'] / 1e6:.1f}"
        ),
        'negative windows': int((values < 0).sum()),
    })

pd.DataFrame(branch_rows).set_index('branch')

## 3. Plot pairwise Fst and the three PBS branches

Each dashed line is the empirical 99th percentile calculated from scaffold 1 only. Red points exceed that threshold. The thicker curve is a five-window running median, included to distinguish sustained regions from isolated noisy windows.

In [ ]:
from pathlib import Path; import warnings
warnings.filterwarnings('ignore', message='Unable to import Axes3D.*')
import matplotlib.pyplot as plt
import pandas as pd

run_dir = Path(__import__('os').environ.get('KENYA2026_WORK_DIR', Path.home() / 'kenya2026')) / 'day4_afternoon_selection_scans' / 'wildebeest_bonus_run'
fst_windows = pd.read_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t')
pbs_windows = pd.read_csv(run_dir / 'scaffold1.pbs.tsv', sep='\t')

def plot_scan(table, panels, y_label, title, output_name):
    position_mb = table['midpoint'] / 1e6
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
    for axis, (column, label, colour) in zip(axes, panels):
        values = table[column]
        threshold = values.quantile(0.99)
        outlier = values >= threshold
        axis.scatter(position_mb, values, s=5, color='0.65', alpha=0.65)
        axis.scatter(position_mb[outlier], values[outlier], s=13, color='#C1292E')
        axis.plot(
            position_mb, values.rolling(5, center=True, min_periods=1).median(),
            color=colour, linewidth=1.3, label='5-window median',
        )
        axis.axhline(threshold, color=colour, linestyle='--', linewidth=1,
                     label=f'scaffold 1 q99 = {threshold:.3f}')
        axis.axhline(0, color='0.82', linewidth=0.7)
        axis.set_ylabel(y_label)
        axis.set_title(label, loc='left')
        axis.legend(loc='lower left', frameon=False, fontsize=8)
    axes[-1].set_xlabel('HiC_scaffold_1 position (Mb)', labelpad=10)
    fig.suptitle(title, fontsize=14)
    fig.savefig(run_dir / output_name, dpi=180, bbox_inches='tight')
    plt.show()

plot_scan(
    fst_windows,
    [
        ('fst_black__maasai_mara', 'Black vs Maasai Mara', '#173F5F'),
        ('fst_black__b_etosha', 'Black vs B-Etosha', '#173F5F'),
        ('fst_maasai_mara__b_etosha', 'Maasai Mara vs B-Etosha', '#173F5F'),
    ],
    'Hudson Fst', 'Pairwise differentiation in non-overlapping 100 kb windows',
    'scaffold1.pairwise_fst.png',
)

plot_scan(
    pbs_windows,
    [
        ('pbs_black', 'Homogeneous black (n=9)', '#222222'),
        ('pbs_maasai_mara', 'Maasai Mara / W-Serengeti (n=10)', '#2F7D32'),
        ('pbs_b_etosha', 'B-Etosha, southern Brindled (n=5)', '#7A5195'),
    ],
    'PBS', 'Population branch statistic in non-overlapping 100 kb windows',
    'scaffold1.pbs.png',
)

### Quick check: interpreting the wildebeest scan

Run the following cell to check the main methodological points before discussing the biological interpretation.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/selection_scans_quizzes/wildebeest_interpretation.json')

## Questions

1. Which pair of populations has the highest background Fst, and why?
2. Compare the locations and shapes of the three PBS signals around 135–137 Mb. What evidence argues against assigning the entire block to selection on one single branch (one single population)?
3. How does the reported black-to-B-Etosha introgression complicate a simple branch-specific selection interpretation?
4. Which follow-up statistics or quality checks would help distinguish selection from reduced diversity, structural variation, or mapping artefacts?

## Compare with the pre-generated whole-genome scans

The following cell loads two pre-generated figures from the shared workshop directory rather than recomputing them. Their dashed thresholds are genome-wide empirical percentiles, whereas the thresholds in the student-generated plots above are scaffold-1 percentiles.

In [ ]:
from pathlib import Path; from IPython.display import Image, Markdown, display

data_dir = Path('selection_wildebeest_bonus')

display(Markdown('### Whole-genome pairwise Fst: black versus B-Etosha'))
display(Image(filename=str(data_dir / 'whole_genome.black_vs_b_etosha.fst.png'), width=1100))

display(Markdown('### Whole-genome PBS: black, Maasai Mara, and B-Etosha'))
display(Image(filename=str(data_dir / 'whole_genome.black_maasai_mara_b_etosha.pbs.png'), width=1100))

## Interpretation and limitations

The scaffold 1 block is not attributable to a single terminal branch. The black branch reaches its maximum at 135.0–135.1 Mb and B-Etosha at 135.7–135.8 Mb; Maasai Mara is also locally elevated. In the whole-genome scan, the strongest Maasai Mara branch window is instead on scaffold 7 at 114.6–114.7 Mb.

This is a candidate-generating scan, not proof of selection. The article inferred ancient black-to-B-Etosha introgression in the broad scaffold 1 region, violating the simple bifurcating-tree interpretation of PBS. Follow-up should examine diversity, $D_{xy}$, LD, missingness, variant density, mapping quality, and gene annotations.

The current VCF is globally MAF-filtered and lightly LD-pruned, so it is not suitable for unbiased Tajima's D or nucleotide-diversity estimation. Shared wildebeest inputs and pre-generated figures are in `selection_wildebeest_bonus/`; intermediate results are written to `wildebeest_bonus_run/`. The notebook implementation agrees with the retained upstream scikit-allel Fst calculation to less than $5\times10^{-16}$.